In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install gym pandas numpy torch scikit-learn matplotlib seaborn

In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Carrega os dados brutos
ratings = pd.read_csv(
    'https://files.grouplens.org/datasets/movielens/ml-100k/u.data',
    sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']
)
print("Ratings:", ratings.shape)

users = pd.read_csv(
    'https://files.grouplens.org/datasets/movielens/ml-100k/u.user',
    sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code']
)
print("Users:", users.shape)

# Lê apenas as colunas necessárias de u.item, ignorando o restante
movies = pd.read_csv(
    'https://files.grouplens.org/datasets/movielens/ml-100k/u.item',
    sep='|', encoding='latin-1', header=None, usecols=[0, 1],
    names=['item_id', 'title']
)
print("Movies:", movies.shape)

# Realiza os merges
data = ratings.merge(users, on='user_id', how='inner')
print("Após merge com users:", data.shape)
data = data.merge(movies, on='item_id', how='inner')
print("Após merge com movies:", data.shape)

# Se estiver vazio aqui, algo deu errado nos merges
if data.empty:
    raise ValueError("O dataframe `data` está vazio após os merges!")

# Codificação
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
data['user'] = user_encoder.fit_transform(data['user_id'])
data['item'] = item_encoder.fit_transform(data['item_id'])

# Fairness por gênero
data['group'] = data['gender'].map({'M': 0, 'F': 1})

# Verificações finais
num_users = data['user'].nunique()
num_items = data['item'].nunique()

print("Usuários únicos codificados:", num_users)
print("Itens únicos codificados:", num_items)

assert num_users > 0 and num_items > 0


Ratings: (100000, 4)
Users: (943, 5)
Movies: (1682, 2)
Após merge com users: (100000, 8)
Após merge com movies: (100000, 9)
Usuários únicos codificados: 943
Itens únicos codificados: 1682


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

In [5]:
# Definir seed para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

In [32]:
import gym
from gym import spaces

class MovieLensEnv(gym.Env):
    def __init__(self, data, num_users, num_items):
        super(MovieLensEnv, self).__init__()
        self.data = data
        self.num_users = num_users
        self.num_items = num_items
        self.action_space = spaces.Discrete(num_items)
        self.observation_space = spaces.Discrete(num_users)
        self.current_user = None

    def reset(self):
        self.current_user = np.random.randint(0, self.num_users)
        return self.current_user

    def step(self, action):
        user_data = self.data[self.data['user'] == self.current_user]
        match = user_data[user_data['item'] == action]
        reward = match['rating'].mean() if not match.empty else 0
        #print(f"[STEP] user={self.current_user}, action={action}, matched_rows={len(match)}, reward={reward}")
        done = True
        info = {'group': user_data['group'].iloc[0] if not user_data.empty else None}
        return self.current_user, reward, done, info


In [25]:
class DQNAgent:
    def __init__(self, num_users, num_items, env, lr=0.001, gamma=0.99):
        self.num_users = num_users
        self.num_items = num_items
        self.env = env
        self.lr = lr
        self.gamma = gamma
        self.model = nn.Sequential(
            nn.Embedding(num_users, 64),
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, num_items)
        )
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()

    def select_action(self, state, epsilon=0.1):
        # Recupera o usuário atual a partir do estado
        user_id = state  # se o estado for só o user_id. Ajuste se for mais complexo.
        
        # Obtém os itens válidos para o usuário
        user_data = self.env.data[self.env.data['user'] == user_id]
        valid_items = user_data['item'].unique()
        
        if len(valid_items) == 0:
            # fallback: caso raro de usuário sem histórico, sorteia qualquer item
            return random.randint(0, self.num_items - 1)

        if random.random() < epsilon:
            # Escolhe aleatoriamente entre os itens avaliados por esse usuário
            return int(np.random.choice(valid_items))
        
        with torch.no_grad():
            q_values = self.model(torch.tensor(state).long().unsqueeze(0))  # [1, num_items]
            q_values = q_values.squeeze(0).detach().cpu().numpy()

            # Filtra os Q-values para apenas os itens válidos
            filtered_q_values = q_values[valid_items]
            best_item_index = np.argmax(filtered_q_values)
            return int(valid_items[best_item_index])


    def train_step(self, state, action, reward, next_state, done):
        self.model.train()
        q_values = self.model(torch.tensor(state).long().unsqueeze(0))
        q_value = q_values[0, action]
        with torch.no_grad():
            next_q_values = self.model(torch.tensor(next_state).long().unsqueeze(0))
            next_q_value = next_q_values.max().item()
            target = reward + (0 if done else self.gamma * next_q_value)
        loss = self.criterion(q_value, torch.tensor(target, dtype=torch.float32))
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()


In [8]:
def compute_group_exposure(agent, data, num_users, num_items):
    exposure = {0: 0, 1: 0}
    counts = {0: 0, 1: 0}
    for user in range(num_users):
        group = data[data['user'] == user]['group'].iloc[0]
        action = agent.select_action(user, epsilon=0)
        exposure[group] += 1 if data[(data['user'] == user) & (data['item'] == action)].shape[0] > 0 else 0
        counts[group] += 1
    exposure_ratio = {g: exposure[g] / counts[g] if counts[g] > 0 else 0 for g in exposure}
    return exposure_ratio


In [9]:
from ml_moo import moo, get_objectives, MooScalarization
from ml_moo.analysis.metrics import compute_hypervolume_progress
from ml_moo.analysis.visualization import plot_pareto_2d, plot_multiple_hypervolumes, plot_hypervolume
from ml_moo.scalarization.rl_scalarization import MovieLensDQNScalarization

In [10]:
num_users = data['user'].nunique()
num_items = data['item'].nunique()

print("Usuários:", num_users)
print("Itens:", num_items)
assert num_users > 0 and num_items > 0

Usuários: 943
Itens: 1682


In [11]:
print(data.columns)
print(data[['item_id', 'item']].drop_duplicates().head())
print(data['item'].nunique())

Index(['user_id', 'item_id', 'rating', 'timestamp', 'age', 'gender',
       'occupation', 'zip_code', 'title', 'user', 'item', 'group'],
      dtype='object')
   item_id  item
0      242   241
1      302   301
2      377   376
3       51    50
4      346   345
1682


In [33]:
num_users = data['user'].nunique()
num_items = data['item'].nunique()
env = MovieLensEnv(data, num_users, num_items)
agent = DQNAgent(num_users, num_items, env)

In [ ]:
scalarization = MovieLensDQNScalarization(agent, env)

# Otimizar com pesos [0.5, 0.5]
scalarization.optimize(np.array([0.5, 0.5]))
print("Recompensas:", scalarization.objs)


w [0.5 0.5] 
[STEP] user=320, action=479, matched_rows=1, reward=4.0
[STEP] user=775, action=647, matched_rows=1, reward=3.0
[STEP] user=653, action=13, matched_rows=1, reward=2.0
[STEP] user=882, action=895, matched_rows=1, reward=5.0
[STEP] user=470, action=392, matched_rows=1, reward=5.0
[STEP] user=142, action=270, matched_rows=1, reward=4.0
[STEP] user=91, action=23, matched_rows=1, reward=3.0
[STEP] user=353, action=250, matched_rows=1, reward=5.0
[STEP] user=833, action=885, matched_rows=1, reward=4.0
[STEP] user=799, action=222, matched_rows=1, reward=5.0
[STEP] user=726, action=23, matched_rows=1, reward=3.0
[STEP] user=853, action=222, matched_rows=1, reward=4.0
[STEP] user=50, action=181, matched_rows=1, reward=3.0
[STEP] user=664, action=185, matched_rows=1, reward=4.0
[STEP] user=697, action=426, matched_rows=1, reward=1.0
[STEP] user=574, action=181, matched_rows=1, reward=3.0
[STEP] user=189, action=332, matched_rows=1, reward=4.0
[STEP] user=313, action=125, matched_row

In [15]:
opt_params = {
    'node_time_limit': 2,
    'target_size': 50,
    'target_gap': 0,
    'node_gap': 0.05,
    'norm': False
}
results = {}

In [36]:
method = 'mola'
scalarization = MovieLensDQNScalarization(agent, env)
moopt = moo(scalarization).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
#hypervolume_values = compute_hypervolume_progress(objs)
hypervolume_values = moopt.get_hypervolumes()
results[method] = {
                "moopt": moopt,
                "objectives": objs,
                "models": moopt.get_models() if hasattr(moopt, "get_models") else None,
                "hypervolume": hypervolume_values
            }

14:17:13 | DEBUG | ml_moo | Filtered parameters for Mola: {'node_time_limit': 2, 'target_size': 50, 'target_gap': 0, 'node_gap': 0.05, 'norm': False}
14:17:13 | DEBUG | ml_moo | Finding 1th individual minimum


w [1. 0.] 


14:17:52 | DEBUG | ml_moo | Finding 2th individual minimum


w [0. 1.] 


14:18:32 | DEBUG | ml_moo | Calculated weights: [0.49850194 0.50149806]
14:18:32 | DEBUG | ml_moo | Importance: -1.6503728383909788e-09
14:18:32 | DEBUG | ml_moo | Global lower bound (y*): [5.  0.5]
14:18:32 | DEBUG | ml_moo | Iteration 1
14:18:32 | DEBUG | ml_moo | Solutions list: [array([5. , 0.5]), array([5. , 0.5])]


w [0.49850194 0.50149806] 


14:19:12 | DEBUG | ml_moo | Calculated weights: [0.49937696 0.50062304]
14:19:12 | DEBUG | ml_moo | Importance: -1.650374059636306e-09
14:19:12 | DEBUG | ml_moo | Global lower bound (y*): [4.  0.5]
14:19:12 | DEBUG | ml_moo | Iteration 2
14:19:12 | DEBUG | ml_moo | Solutions list: [array([4. , 0.5])]


new solution added [4.  0.5]
solution dominated [5.  0.5]
solution dominated [5.  0.5]
w [0.49937696 0.50062304] 


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

weights = np.linspace(0, 1, 11)
rewards = []
fairness = []

for w in weights:
    scalarization.optimize([w, 1 - w])
    rewards.append(scalarization.objs[0])
    fairness.append(scalarization.objs[1])

plt.plot(weights, rewards, label='Recompensa')
plt.plot(weights, fairness, label='Equidade')
plt.xlabel('Peso da Recompensa')
plt.ylabel('Valor')
plt.legend()
plt.title('Trade-off entre Recompensa e Equidade')
plt.show()
